# TinyCUA Agent Trace Demo

Use this notebook to run the TinyCUA agent against a local OpenAI Chat Completions-compatible LLM server and inspect which nodes ran, which route labels were selected, and which tools were exposed to each node.

Expected local setup:

- Base URL: `http://localhost:1234/v1`
- Model: `qwen/qwen3.5-4b`
- API key: any non-empty dummy string, e.g. `tinycua-local-test`

Run from `src/tinycua` with:

```bash
uv run jupyter notebook notebooks/tinycua_agent_trace_demo.ipynb
```


## 1. Setup

This cell loads `.env.test` when present and builds helpers for creating agents, running prompts, and printing the execution trace.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import dotenv
from tinycua_sdk.agent.llm_model import LanguageModel

from tinycua.factory import create_tinycua_agent

PROJECT_DIR = Path.cwd()
ENV_FILE = PROJECT_DIR / '.env.test'
dotenv.load_dotenv(ENV_FILE, override=False)

BASE_URL = os.environ.get('OPENAI_CHAT_COMPLETIONS_BASE_URL', 'http://localhost:1234/v1')
MODEL = os.environ.get('OPENAI_CHAT_COMPLETIONS_MODEL', 'qwen/qwen3.5-4b')
API_KEY = os.environ.get('OPENAI_CHAT_COMPLETIONS_API_KEY', 'tinycua-local-test')

print(f'Using base_url={BASE_URL}')
print(f'Using model={MODEL}')
print(f'Loaded env file: {ENV_FILE.exists()} ({ENV_FILE})')

In [ ]:
def make_agent():
    """Create a fresh TinyCUA agent backed by the local LLM."""
    return create_tinycua_agent(
        llm_model=LanguageModel(
            provider='openai-chat-completions',
            model_name=MODEL,
            base_url=BASE_URL,
            api_key=API_KEY,
        )
    )


def show_trace(agent):
    """Pretty-print the loop execution trace."""
    trace = agent.loop.get_execution_trace()
    if not trace:
        print('No trace recorded yet. Run the agent first.')
        return trace

    for index, step in enumerate(trace, start=1):
        print(f'[{index}] {step.get("node_id")} ({step.get("node_type")})')
        print(f'    route: {step.get("route_label")}')
        print(f'    terminal: {step.get("is_terminal")}')
        print(f'    tools: {step.get("resolved_tool_names", [])}')
        print()
    return trace


async def run_prompt(prompt: str, *, stream: bool = False):
    """Run a prompt and show result + trace."""
    agent = make_agent()

    if stream:
        stream_iter = await agent.run(prompt, stream=True)
        chunks = []
        events = []
        async for event in stream_iter:
            events.append(event)
            if event.get('type') == 'response.output_text.delta':
                chunks.append(event.get('delta', ''))
        result = ''.join(chunks)
        print(f'Collected {len(events)} streaming events')
    else:
        result = await agent.run(prompt)

    print('=== RESULT ===')
    print(result)
    print('\n=== TRACE ===')
    trace = show_trace(agent)
    return agent, result, trace

## 2. Passthrough / simple prompt

This verifies that the public factory starts at `query_analyst`, exposes `select_query_route`, and eventually reaches `response`. The live model may choose `passthrough`, `uncertain`, or even `worker`; the goal is to inspect the node trace and tool exposure.

In [ ]:
agent, result, trace = await run_prompt('Say hello in one short sentence.')

## 3. Worker-mode prompt

This prompt is intentionally task-like so you can inspect the worker path. Expected interesting nodes include `query_analyst`, `digester`, `worker`, and `response`.

In [ ]:
worker_prompt = '''
Can you make me an app to take notes and schedule so I can keep track
of everything where backend should use Python and frontend with web UI?
'''

agent, result, trace = await run_prompt(worker_prompt)

## 4. Streaming run

This checks the streaming path and still prints the final trace after the stream is consumed.

In [ ]:
agent, result, trace = await run_prompt(worker_prompt, stream=True)

## 5. Raw trace JSON

Use this if you want to copy/paste the trace into an issue or review.

In [ ]:
print(json.dumps(trace, indent=2, default=str))

## Troubleshooting

- If imports fail, start Jupyter from `src/tinycua` using `uv run jupyter notebook`.
- If the model cannot connect, confirm your local OpenAI-compatible server is running at `http://localhost:1234/v1`.
- If the server does not require an API key, still pass a non-empty dummy key because some code paths validate that a key exists.
- If a simple prompt routes to `worker`, that can happen with live models. The important debugging signal is whether `select_query_route` was exposed and which nodes actually ran.